# Socio-economic indicators evaluation

Read aggregated blocks spatial layer

In [1]:
import pandas as pd

blocks_gdf = pd.read_pickle('blocks.pickle')

In [2]:
service_types_df = pd.read_pickle('service_types.pickle')

In [3]:
st_df = service_types_df[~service_types_df.blocksnet.isna()].copy()
st_df['service_type_id'] = st_df.index
agg_df = st_df.groupby('blocksnet').agg({'service_type_id': lambda s : list(s)})
new_columns = {}
for st_name, row in agg_df.iterrows():
    st_ids = row['service_type_id']
    for prefix in ['count', 'capacity']:
        sum_df = blocks_gdf[[f'{prefix}_{st_id}' for st_id in st_ids]].sum(axis=1)
        new_columns[f'{prefix}_{st_name}'] = sum_df
new_columns_df = pd.DataFrame.from_dict(new_columns)
blocks_gdf = pd.concat([blocks_gdf, new_columns_df], axis=1)

Read drive graph

In [4]:
from blocksnet.relations import get_accessibility_graph

graph = get_accessibility_graph(blocks_gdf, 'drive')

2025-10-15 04:39:07.650 | INFO     | iduedu.modules.drive_walk_builder:get_drive_graph_by_poly:91 - Downloading drive graph from OSM, it may take a while for large territory ...


Read drive accessibility matrix

In [5]:
acc_mx = pd.read_pickle('acc_mx.pickle')

## General indicators

In [6]:
from blocksnet.analysis.indicators.socio_economic import calculate_general_indicators

general_indicators = calculate_general_indicators(blocks_gdf)
general_indicators

,total
Area (km2),1796.717693
Urbanization,0.002227


## Demographic indicators

In [7]:
from blocksnet.analysis.indicators.socio_economic import calculate_demographic_indicators

demographic_indicators = calculate_demographic_indicators(blocks_gdf.fillna(0).rename(columns={'territory_id':'parent'}))
demographic_indicators

,117,121,total
Population,14862.000000,279.000000,15141.000000
Density (people/km2),20.978483,0.256368,8.427033


## Settlement indicators

In [8]:
# not calculated within blocksnet

## Transport indicators

In [9]:
from blocksnet.analysis.indicators.socio_economic import calculate_transport_indicators

transport_indicators = calculate_transport_indicators(blocks_gdf.rename(columns={'territory_id':'parent'}), acc_mx, graph)
transport_indicators

,117,121,total
Road network density (km/km2),0.530243,0.355992,0.424698
Settlements connectivity,0.258009,NaN,0.258009
Road network length (km),375.645219,387.417992,763.063211
Fuel stations count,5.000000,0.000000,5.000000
Average fuel station accessibility (min),36.193548,NaN,36.193548
Railway stops count,8.000000,4.000000,12.000000
Average railway stop accessibility (min),38.619355,NaN,38.619355
Airports count,1.000000,0.000000,1.000000
Average airport accessibility (min),24.412903,NaN,24.412903


## Economic indicators

In [10]:
# not calculated within blocksnet

## Ecological indicators

In [11]:
# not calculated within blocksnet

## Engineering indicators

In [12]:
from blocksnet.analysis.indicators.socio_economic import calculate_engineering_indicators

blocks_gdf = blocks_gdf.copy()
blocks_gdf[['count_reservoir', 'count_gas_distribution']] = 0.0

engineering_indicators = calculate_engineering_indicators(blocks_gdf)
engineering_indicators

,total
Substation,0
Water works,0
Wastewater plant,0
Reservoir,0
Gas distribution,0
Infrastructure object,0


## Social indicators

In [13]:
from blocksnet.analysis.indicators.socio_economic import SocialIndicator

SOCIAL_COUNT_INDICATORS_MAPPING = {
    SocialIndicator.EXTRACURRICULAR: [23, 24],
    SocialIndicator.AMBULANCE: [39, 40],
    SocialIndicator.SPECIAL_MEDICAL: [41],
    SocialIndicator.PREVENTIVE_MEDICAL: [42],
    SocialIndicator.GYM: [68],
    SocialIndicator.ORPHANAGE: [46],
    SocialIndicator.SOCIAL_SERVICE_CENTER: [43],
    SocialIndicator.CULTURAL_CENTER: [49],
    SocialIndicator.CONCERT_HALL: [53],
    SocialIndicator.ICE_ARENA: [60],
    SocialIndicator.ECO_TRAIL: [72],
    SocialIndicator.FIRE_STATION: [79],
    SocialIndicator.TOURIST_BASE: [112],
}

In [14]:
from blocksnet.analysis.indicators.socio_economic import calculate_social_indicators

count_indicators, provision_indicators = calculate_social_indicators(blocks_gdf.fillna(0), acc_mx)

100%|██████████| 39/39 [00:01<00:00, 34.79it/s]


In [15]:
count_indicators

,total
Kindergarten,8.0
School,5.0
College,1.0
University,0.0
Extracurricular,NaN
Hospital,1.0
Polyclinic,1.0
Ambulance,NaN
Sanatorium,0.0
Special medical,NaN


In [16]:
provision_indicators

,total
Kindergarten,0.779643
School,0.739837
College,0.898734
University,0.000000
Extracurricular,NaN
Hospital,0.011905
Polyclinic,0.012876
Ambulance,NaN
Sanatorium,0.000000
Special medical,NaN
